# Russian River Step 3 -- 2D mesh

Form the 2D mesh, elevate via a DEM, condition.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
# setting up logging first or else it gets preempted by another package
import watershed_workflow.io
watershed_workflow.io.setupLogging(1)

In [ ]:
import os,sys
import logging
import numpy as np
from matplotlib import pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

import pickle
import shapely
import pandas as pd
import geopandas as gpd
pd.options.display.max_columns = None
import copy

import watershed_workflow 
import watershed_workflow.utils
import watershed_workflow.utils.geometry
import watershed_workflow.sources
import watershed_workflow.mesh
import watershed_workflow.plot
import watershed_workflow.sources.standard_names

# set the default figure size for notebooks
plt.rcParams["figure.figsize"] = (8, 6)

## Input: Parameters and other source data

In [ ]:
# Force Watershed Workflow to pull data from this directory rather than a shared data directory.
# This picks up the Coweeta-specific datasets set up here to avoid large file downloads for 
# demonstration purposes.
#
def splitPathFull(path):
    """
    Splits an absolute path into a list of components such that
    os.path.join(*splitPathFull(path)) == path
    """
    parts = []
    while True:
        head, tail = os.path.split(path)
        if head == path:  # root on Unix or drive letter with backslash on Windows (e.g., C:\)
            parts.insert(0, head)
            break
        elif tail == path:  # just a single file or directory
            parts.insert(0, tail)
            break
        else:
            parts.insert(0, tail)
            path = head
    return parts

cwd = splitPathFull(os.getcwd())
assert cwd[-1] == 'workflow'
cwd = cwd[:-1]

# Note, this directory is where downloaded data will be put as well
data_dir = os.path.join(*(cwd + ['input_data',]))
def toInput(filename):
    return os.path.join(data_dir, filename)

output_dir = os.path.join(*(cwd + ['output_data',]))
output_filenames = dict()
def fromOutput(filename):
    return os.path.join(output_dir, filename)    

def toOutput(role, filename):
    output_filenames[role] = filename
    return fromOutput(filename)

# check output and input dirs exist
if not os.path.isdir(data_dir):
    os.makedirs(data_dir, exist_ok=True)
if not os.path.isdir(output_dir):
    os.makedirs(output_dir, exist_ok=True)
       

In [ ]:
# Set the data directory to the local space to get the locally downloaded files
# REMOVE THIS CELL for general use outside fo Coweeta
watershed_workflow.utils.setDataDirectory(data_dir)


In [ ]:
## Parameters cell -- this provides all parameters that can be changed via pipelining to generate a new watershed. 
name = 'RussianRiver'
hucs = ['18010110'] # a list of HUCs to run


# -- parameters to clean and reduce the river network prior to meshing
prune_by_area = 20               # km^2
simplify = 200                   # length scale to target average edge 

# -- mesh triangle refinement control
refine_d0 = 200
refine_d1 = 600

refine_L0 = 200
refine_L1 = 500

refine_A0 = refine_L0**2 / 2
refine_A1 = refine_L1**2 / 2


# Refine triangles if they get too acute
min_angle = 20 # degrees

# width of reach by stream order (order:width)
river_widths = dict({1:10, 2:10, 3:20, 4:30, 5:30}) 


# Note that, by default, we tend to work in the DayMet CRS because this allows us to avoid
# reprojecting meteorological forcing datasets.
crs = watershed_workflow.crs.default_crs

## Reload data

In [ ]:
with open(fromOutput('03a_watersheds.pickle'), 'rb') as fid:
    watersheds = pickle.load(fid)

reaches = gpd.read_parquet(fromOutput('03b_rivers.parquet'))
rivers = watershed_workflow.hydro.createRivers(reaches, method='native')


In [ ]:
# load m2 mesh from file -- we will use this for testing pitfilling algorithms

with open(fromOutput('03c_m2.pickle'), 'rb') as fid:
    m2 = pickle.load(fid)

In [ ]:
# load the gage locations
gages = gpd.read_parquet(fromOutput('01_gages.parquet'))

In [ ]:
gages['name']


In [ ]:
gages['discrete_location'] = [shapely.geometry.Point(0.,0.) for g in gages.index]

for (index, comid) in zip(gages.index, gages.comid):
    try:
        reach = next(r for r in rivers[0] if r['comid'] == comid)
    except StopIteration:
        print(f'Cannot find comid {comid}')
        gages.loc[index, 'discrete_location'] = None
        continue

    gage_p = gages.loc[index, 'geometry']
    nearest_p, _ = shapely.ops.nearest_points(reach.linestring, gage_p)
    nearest_p_dist = watershed_workflow.utils.geometry.computeDistance(nearest_p.coords[0], gage_p.coords[0])

    def dist(i_coord):
        i, coord = i_coord
        d = watershed_workflow.utils.geometry.computeDistance(coord, gage_p.coords[0])
        print(i,d)
        return d
    
    nearest_discrete_p_i, nearest_discrete_p = min(enumerate(reach.linestring.coords),
                               key=dist)
    nearest_discrete_p_dist = watershed_workflow.utils.geometry.computeDistance(nearest_discrete_p, gage_p.coords[0])

    print('')
    print(f'Gage: {gages.loc[index, 'name']}')
    print('------------------------')
    print(f'comid: {comid}')
    print(f'Nearest point: {nearest_p} at {nearest_p_dist} m away')
    print(f'Nearest discrete point: {nearest_discrete_p} at {nearest_discrete_p_dist} m away')
    print(f'   is coord {nearest_discrete_p_i} of {len(reach.linestring.coords)}')

    coords = np.array(list(reversed(reach.linestring.coords)))
    diffs = np.diff(coords, axis=0)
    cumulative_dists = np.concatenate([[0,], np.cumsum(np.sqrt((diffs ** 2).sum(axis=1)))])
    assert cumulative_dists[0] == 0.
    if len(list(reach.siblings)) > 0 and cumulative_dists[nearest_discrete_p_i] < 500:
        if cumulative_dists[-1] < 500:
            nearest_discrete_p_i = len(coords) - 1
        else:
            nearest_discrete_p_i = np.where(cumulative_dists > 500)[0][0]

        nearest_discrete_p = coords[nearest_discrete_p_i]
        print(f'Moving point to new coordinate {nearest_discrete_p_i} at {nearest_discrete_p}')

    # found the point, now add the region
    gages.loc[index, 'discrete_location'] = shapely.geometry.Point(nearest_discrete_p)

    
# add the regions
gages_found = gages[~gages['discrete_location'].isna()]
watershed_workflow.mesh.addDischargeRegions(m2, 
                                            gages_found['discrete_location'].values, 
                                            [f'USGS-{name}' if not name.startswith('RR') else name for name in gages_found['name']],
                                            )
gages_found
    

In [ ]:
gages_found['discrete_location'] = gages_found['discrete_location'].astype('geometry')
gages_found.to_parquet(toOutput('03d_gages_found', '03d_gages_found.parquet'))

In [ ]:
print("labeled sets")
print(f"label\tname\tcount")
for ls in m2.labeled_sets:
    print(f"{ls.setid}\t{ls.name}\t{len(ls.ent_ids)}")

In [ ]:
# save the mesh and regions
with open(toOutput('m2', '03d_m2.pickle'), 'wb') as fid:
    pickle.dump(m2, fid)

In [ ]:
# save output filenames
with open(toOutput('03d_output_filenames', '03d_output_filenames.txt'), 'wb') as fid:
    pickle.dump(output_filenames, fid)